In [36]:
import ROOT
import tensorflow as tf
import numpy as np


tree_name = "myTree"
root_file_list = ["temp0_1.root"]
type1_branches = ['MET_cal1_sumpt', 'jet1_pt', 'jet2_pt', 'jet3_pt', 'jet4_pt', 'photon1_pt']
type2_branches = ['jet1_phi', 'jet2_phi', 'jet3_phi', 'jet4_phi', 'photon1_phi']
type3_branches = ['jet1_eta', 'jet2_eta', 'jet3_eta', 'jet4_eta', 'photon1_eta']
type4_branches =  ['MET_cal1_e', 'photon1_topoetcone20', 'photon1_topoetcone40']
type5_branches =  ['MET_cal1_px', 'MET_cal1_py', 'MET_cal1_pz']
branches = type1_branches + type2_branches + type3_branches + type4_branches + type5_branches 
# Load RDataFrame
#rdf = ROOT.RDF.Experimental.Distributed.Dask.RDataFrame( tree_name , root_file_list, daskclient=client, npartitions=200 )
rdf = ROOT.RDataFrame(tree_name, root_file_list)
batch_size = 1024
chunk_size = 10000

ds_train, ds_valid = ROOT.TMVA.Experimental.CreateTFDatasets(
    rdf,
    batch_size=batch_size,
    chunk_size=chunk_size,
    target=[],  # <- Use an empty list for unsupervised
    validation_split=0.1
)

In [37]:
ds_train

<_FlatMapDataset element_spec=(TensorSpec(shape=(1024, 22), dtype=tf.float32, name=None), TensorSpec(shape=(1024, 0), dtype=tf.float32, name=None))>

In [38]:
"""
ds_train = ds_train.map(lambda x, y: (tf.expand_dims(x, axis=-1), y))
ds_valid = ds_valid.map(lambda x, y: (tf.expand_dims(x, axis=-1), y))
"""

'\nds_train = ds_train.map(lambda x, y: (tf.expand_dims(x, axis=-1), y))\nds_valid = ds_valid.map(lambda x, y: (tf.expand_dims(x, axis=-1), y))\n'

In [39]:
import tensorflow as tf
from tensorflow.keras import layers, Model , Sequential, losses ,metrics, optimizers
from tensorflow.keras.layers import LeakyReLU, Input, Dense, LayerNormalization, Dropout,Lambda, Concatenate, Layer, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.nn import softmax
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tensorflow.keras.callbacks import LearningRateScheduler

In [52]:
# Custom Layer Normalization
class CustomLayerNormalization(layers.Layer):
    def __init__(self, epsilon=1e-6):
        super(CustomLayerNormalization, self).__init__()
        self.epsilon = epsilon

    def build(self, input_shape):
        self.gamma = self.add_weight(shape=(input_shape[-1],), initializer="ones", trainable=True)
        self.beta = self.add_weight(shape=(input_shape[-1],), initializer="zeros", trainable=True)

    def call(self, inputs):
        mean = tf.reduce_mean(inputs, axis=-1, keepdims=True)
        variance = tf.reduce_mean(tf.square(inputs - mean), axis=-1, keepdims=True)
        normalized = (inputs - mean) / tf.sqrt(variance + self.epsilon)
        return self.gamma * normalized + self.beta


# Custom Linear Layer with Weight Decay
class CustomLinear(layers.Layer):
    def __init__(self, out_features, weight_decay=0.00001):
        super(CustomLinear, self).__init__()
        self.out_features = out_features
        self.weight_decay = weight_decay

    def build(self, input_shape):
        self.weight = self.add_weight(shape=(input_shape[-1], self.out_features),
                                      initializer="random_normal",
                                      regularizer=tf.keras.regularizers.l2(self.weight_decay),
                                      trainable=True)
        self.bias = self.add_weight(shape=(self.out_features,), initializer="zeros", trainable=True)

    def call(self, inputs):
        return tf.matmul(inputs, self.weight) + self.bias


# Advanced Feed Forward Layer
class AdvancedFeedForward(layers.Layer):
    def __init__(self, dff, d_model, rate, weight_decay, alpha_rt):
        super(AdvancedFeedForward, self).__init__()
        self.dense1 = CustomLinear(dff, weight_decay)
        self.dense2 = CustomLinear(d_model, weight_decay)
        self.dropout = layers.Dropout(rate)
        self.layernorm = layers.BatchNormalization()
        self.alpha_rt = alpha_rt

    def call(self, x, training):
        x = self.dense1(x)
        x = tf.nn.leaky_relu(x, alpha=self.alpha_rt)
        x = self.dropout(x, training=training)
        x = self.dense2(x)
        return self.layernorm(x + self.dropout(x, training=training))


# Multi-Head Attention Layer
class MultiHeadAttention(layers.Layer):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        assert d_model % self.num_heads == 0
        self.depth = d_model // self.num_heads
        self.wq = layers.Dense(d_model)
        self.wk = layers.Dense(d_model)
        self.wv = layers.Dense(d_model)
        self.dense = layers.Dense(d_model)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def scaled_dot_product_attention(self, q, k, v, mask=None):
        matmul_qk = tf.matmul(q, k, transpose_b=True)
        dk = tf.cast(tf.shape(k)[-1], tf.float32)
        scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)
        attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
        output = tf.matmul(attention_weights, v)
        return output, attention_weights

    def call(self, q, k, v, mask=None):
        batch_size = tf.shape(q)[0]
        q = self.split_heads(self.wq(q), batch_size)
        k = self.split_heads(self.wk(k), batch_size)
        v = self.split_heads(self.wv(v), batch_size)
        scaled_attention, attention_weights = self.scaled_dot_product_attention(q, k, v, mask)
        scaled_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(scaled_attention, (batch_size, -1, self.d_model))
        return self.dense(concat_attention), attention_weights


# Transformer Block
class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, dff, rate, weight_decay, alpha_rt):
        super(TransformerBlock, self).__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = AdvancedFeedForward(dff, d_model, rate, weight_decay, alpha_rt)
        self.layernorm1 = layers.BatchNormalization(epsilon=1e-6)
        self.layernorm2 = layers.BatchNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, x, training, mask=None):
        attn_output, attention_weights = self.mha(x, x, x, mask=mask)
        out1 = self.layernorm1(x + self.dropout1(attn_output))
        ffn_output = self.ffn(out1, training=training)
        out2 = self.layernorm2(out1 + self.dropout1(ffn_output))
        return out2, attention_weights


# Transformer Encoder
class TransformerEncoder(layers.Layer):
    def __init__(self, num_layers, d_model, num_heads, dff, rate, weight_decay, alpha_rt):
        super(TransformerEncoder, self).__init__()
        self.num_layers = num_layers
        self.enc_layers = [TransformerBlock(d_model, num_heads, dff, rate, weight_decay, alpha_rt) for _ in range(num_layers)]

    def call(self, x, training=None, mask=None):
        attention_weights_all_layers = []
        for i in range(self.num_layers):
            x, attention_weights = self.enc_layers[i](x, training=training)
            attention_weights_all_layers.append(attention_weights)
        return x, attention_weights_all_layers


# Transformer Decoder
class TransformerDecoder(layers.Layer):
    def __init__(self, num_layers, d_model, num_heads, dff, rate, weight_decay, alpha_rt):
        super(TransformerDecoder, self).__init__()
        self.num_layers = num_layers
        self.dec_layers = [TransformerBlock(d_model, num_heads, dff, rate, weight_decay, alpha_rt) for _ in range(num_layers)]

    def call(self, x, enc_output, training=None, mask=None):
        attention_weights_all_layers = []
        for i in range(self.num_layers):
            x, attention_weights = self.dec_layers[i](x, training=training)
            attention_weights_all_layers.append(attention_weights)
        return x, attention_weights_all_layers


# Transformer Model
class Transformer(Model):
    def __init__(self, num_encoder_layers, num_decoder_layers, input_dim, d_model, num_heads, dff, rate, weight_decay, alpha_rt):
        super(Transformer, self).__init__()
        self.encoder = TransformerEncoder(num_encoder_layers, d_model, num_heads, dff, rate, weight_decay, alpha_rt)
        self.decoder = TransformerDecoder(num_decoder_layers, d_model, num_heads, dff, rate, weight_decay, alpha_rt)
        self.final_layer = layers.Dense(1)
        self.d_model = d_model
        self.input_embedding = tf.keras.layers.Dense(d_model)
  

    def get_config(self):
        config = super(Transformer, self).get_config()
        config.update({
            'num_encoder_layers': self.encoder.num_layers,
            'num_decoder_layers': self.decoder.num_layers,
            'd_model': self.d_model,
            'num_heads': self.encoder.enc_layers[0].mha.num_heads,
            'dff': self.encoder.enc_layers[0].ffn.dense1.out_features,
            'input_dim': self.final_layer.units,
            'rate': self.encoder.enc_layers[0].ffn.dropout.rate,
            'weight_decay': self.encoder.enc_layers[0].ffn.dense1.weight_decay,
            'alpha_rt': self.encoder.enc_layers[0].ffn.alpha_rt, # Get alpha_rt from AdvancedFeedForward
        })
        return config

    @classmethod
    def from_config(cls, config, custom_objects=None):
        # Extract and remove Keras-specific attributes before passing to __init__
        return cls(
            num_encoder_layers=config['num_encoder_layers'],
            num_decoder_layers=config['num_decoder_layers'],
            d_model=config['d_model'],
            num_heads=config['num_heads'],
            dff=config['dff'],
            input_dim=config['input_dim'],
            rate=config['rate'],
            weight_decay=config['weight_decay'],
            alpha_rt=config['alpha_rt']
        )

    def call(self, inputs, training=None):
        #input_= tf.expand_dims(inputs, axis=-1)
        #enc_input= self.input_embedding(inputs)
        
        enc_output, _ = self.encoder(inputs, training=training)
        dec_input = enc_input
        final_output, _ = self.decoder(dec_input, enc_output, training=training)
        final_output = self.final_layer(final_output)
        return final_output


class LossHistory(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs={}):
        self.losses = []
        self.val_losses = []

    def on_epoch_end(self, batch, logs={}):
        self.losses.append(logs.get('loss'))
        self.val_losses.append(logs.get('val_loss'))

    def save_losses(self, file_path):
        with open(file_path, 'w') as file:
            json.dump({'losses': self.losses, 'val_losses': self.val_losses}, file)

    def load_losses(self, file_path):
        with open(file_path, 'r') as file:
            data = json.load(file)
            self.losses = data['losses']
            self.val_losses = data['val_losses']
loss_history = LossHistory()


class EarlyStopping(tf.keras.callbacks.Callback):
    def __init__(self, patience=5, min_delta=0):
        super(EarlyStopping, self).__init__()
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.best_weights = None

    def on_epoch_end(self, epoch, logs=None):
        current_loss = logs.get("val_loss")
        if self.best_loss is None:
            self.best_loss = current_loss
            self.best_weights = self.model.get_weights()  # Save weights
        elif self.best_loss - current_loss > self.min_delta:
            self.best_loss = current_loss
            self.best_weights = self.model.get_weights()  # Save new best weights
            self.counter = 0
        else:
            self.counter += 1
            print(f"INFO: Early stopping counter {self.counter} of {self.patience}")
            if self.counter >= self.patience:
                print('INFO: Early stopping')
                self.model.stop_training = True
                self.model.set_weights(self.best_weights)  # Restore best weights



import tensorflow as tf
from tensorflow.keras import backend as K

class CustomWeightedLoss(tf.keras.losses.Loss):
    def __init__(self, weights, name="custom_weighted_loss"):
        super().__init__(name=name)
        self.weights = tf.constant(weights, dtype=tf.float32)

    def call(self, y_true, y_pred):
        # Compute element-wise squared error (example)
        squared_error = tf.square(y_true - y_pred)
        # Apply weights broadcasting over last dimension (adjust as needed)
        weighted_error = squared_error * self.weights
        # Reduce over last dimension (e.g., features)
        loss = tf.reduce_mean(weighted_error, axis=-1)
        # Return mean loss over batch
        return tf.reduce_mean(loss)

        

In [50]:
# Assuming ds_train and ds_valid are the TF datasets from CreateTFDatasets
# Map them to input = target pairs for reconstruction
ds_train_recon = ds_train.map(lambda x, y: (x, x))
ds_valid_recon = ds_valid.map(lambda x, y: (x, x))

# Define your scheduler callback
def scheduler(epoch):
    if epoch < 10:
        return 1e-3
    elif epoch < 20:
        return 1e-4
    elif epoch < 30:
        return 1e-5
    else:
        return 5e-6

lr_scheduler = tf.keras.callbacks.LearningRateScheduler(scheduler)

# Define EarlyStopping callback
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Now compile your model if not done already
input_dim=len(branches)
weight_loss= [1]*len(branches)

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
              loss='mse')

# Train using datasets
history = model.fit(
    ds_train_recon,
    validation_data=ds_valid_recon,
    epochs=100,
    callbacks=[lr_scheduler, loss_history, early_stopping],
    verbose=1,
)


Epoch 1/100


ValueError: Dimensions must be equal, but are 22 and 1024 for '{{node compile_loss/mse/sub}} = Sub[T=DT_FLOAT](data_1, compile_loss/mse/Squeeze)' with input shapes: [1024,22], [1024,1024].